# CHAPTER 8 : Data Wrangling: Join, Combine, and Reshape

-----

In many applications, data may be spread across a number of files or databases or be
arranged in a form that is not easy to analyze. This chapter focuses on tools to help
combine, join, and rearrange data.

First, I introduce the concept of hierarchical indexing in pandas, which is used extensively
in some of these operations. I then dig into the particular data manipulations.
You can see various applied usages of these tools in Chapter 14.

## Basic Imports and Set ups

In [4]:
import pandas as pd
import numpy as np
import os
from faker import Faker

This is a function to render dataframes as tables in the Jupyter Notebook.

In [5]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)



## 8.1 Hierarchical Indexing

*Hierarchical indexing* is an important feature of pandas that enables you to have multiple
(two or more) index levels on an axis. Somewhat abstractly, it provides a way for
you to work with higher dimensional data in a lower dimensional form. Let’s start
with a simple example; create a Series with a list of lists (or arrays) as the index:

In [6]:
data = pd.Series(np.random.randn(9),index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],[1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [7]:
data # You can see 2 indexes for Series.

a  1   -0.688790
   2    0.793909
   3    0.873270
b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
d  2    0.474364
   3   -0.680989
dtype: float64

What you’re seeing is a prettified view of a Series with a MultiIndex as its index. The
“gaps” in the index display mean “use the label directly above”:

In [8]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [13]:
data.index.levels

FrozenList([['a', 'b', 'c', 'd'], [1, 2, 3]])

In [12]:
data.index.codes  # Codes was earlier called Labels.

FrozenList([[0, 0, 0, 1, 1, 2, 2, 3, 3], [0, 1, 2, 0, 2, 0, 1, 1, 2]])

With a hierarchically indexed object, so-called partial indexing is possible, enabling
you to concisely select subsets of the data

In [9]:
data['b']

1    1.889498
3   -0.951723
dtype: float64

---------

##### ✅ What does **`labels`** mean in this `MultiIndex`?

When you create a **`MultiIndex`**, pandas internally stores it in a **factorized (encoded) form** to save memory and improve performance.

It does **NOT store the actual index values repeatedly.**
Instead, it stores:

* **`levels` → the unique values at each index level**
* **`labels` → integer pointers (positions) telling which level value to use for each row**

---

##### 🔵 In your example

You created:

```python
data = pd.Series(
    np.random.randn(9),
    index=[
        ['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
        [1, 2, 3, 1, 3, 1, 2, 2, 3]
    ]
)
```

And pandas shows:

```text
MultiIndex(
  levels=[['a','b','c','d'], [1,2,3]],
  labels=[[0,0,0,1,1,2,2,3,3], [0,1,2,0,2,0,1,1,2]]
)
```

---

##### 🟢 Step-1: Understand `levels`

These are the **unique sorted values** for each index level.

Level-0:

```
['a','b','c','d']
```

Level-1:

```
[1,2,3]
```

---

##### 🟢 Step-2: Understand `labels`

`labels` are **integer arrays that reference positions inside the `levels` list.**

Think of them as:

> “Which value from `levels` should be used at this row?”

---

###### ⭐ Level-0 labels

```
[0,0,0,1,1,2,2,3,3]
```

Mapping using:

```
levels[0] = ['a','b','c','d']
```

So:

| Label | Means |
| ----- | ----- |
| 0 →   | 'a'   |
| 1 →   | 'b'   |
| 2 →   | 'c'   |
| 3 →   | 'd'   |

Thus pandas reconstructs:

```
a a a b b c c d d
```

---

###### ⭐ Level-1 labels

```
[0,1,2,0,2,0,1,1,2]
```

Mapping using:

```
levels[1] = [1,2,3]
```

So:

| Label | Means |
| ----- | ----- |
| 0 →   | 1     |
| 1 →   | 2     |
| 2 →   | 3     |

Thus pandas reconstructs:

```
1 2 3 1 3 1 2 2 3
```

---

##### 🎯 Final Reconstruction Logic

For row `i`:

```
actual_index_value =
(
   levels[0][ labels[0][i] ],
   levels[1][ labels[1][i] ]
)
```

Example → row 4

```
labels → (1 , 2)

levels[0][1] → 'b'
levels[1][2] → 3

→ index = ('b', 3)
```

---

##### 🚀 Why pandas does this (VERY important Data Science insight)

This is basically **categorical / dictionary encoding of index values.**

Benefits:

✅ avoids storing repeated strings
✅ faster grouping / sorting
✅ lower memory
✅ vectorized operations on integer codes

This exact idea appears in:

* pandas `Categorical`
* database dictionary encoding
* parquet column encoding
* feature engineering pipelines

---

##### ⚠️ Modern pandas note (important)

In **new pandas versions**, `labels` is renamed to:

```
codes
```

So you may see:

```python
data.index.codes
```

instead of:

```
data.index.labels
```

Same concept. Only name changed.

---



With a hierarchically indexed object, so-called partial indexing is possible, enabling
you to concisely select subsets of the data:

In [14]:
data['b']

1    1.889498
3   -0.951723
dtype: float64

In [15]:
data['b':'c']

b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
dtype: float64

In [16]:
data.loc[['b', 'd']]

b  1    1.889498
   3   -0.951723
d  2    0.474364
   3   -0.680989
dtype: float64

Selection is even possible from an “inner” level:

In [18]:
data.loc[:, 2]

a    0.793909
c   -0.152598
d    0.474364
dtype: float64

Hierarchical indexing plays an important role in reshaping data and group-based
operations like forming a pivot table. For example, you could rearrange the data into
a DataFrame using its `unstack` method:

In [19]:
data.unstack()

,1,2,3
a,-0.688790,0.793909,0.873270
b,1.889498,NaN,-0.951723
c,-0.149447,-0.152598,NaN
d,NaN,0.474364,-0.680989


The inverse operation of `unstack` is `stack`:

In [21]:
data.unstack().stack()

a  1   -0.688790
   2    0.793909
   3    0.873270
b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
d  2    0.474364
   3   -0.680989
dtype: float64

`stack` and `unstack`will be explored in more detail later in this chapter.

----

In [55]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],columns=[['Ohio', 'Ohio', 'Colorado'],['Green', 'Red', 'Green']])

With a DataFrame, either axis can have a hierarchical index:

In [56]:
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

The hierarchical levels can have names (as strings or any Python objects). If so, these
will show up in the console output:

In [57]:
frame.index.names = ['key1', 'key2']

In [58]:
frame.columns.names = ['state', 'color']

In [59]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

> Be careful to distinguish the index names 'state' and 'color'
from the row labels.

With partial column indexing you can similarly select groups of columns:

In [60]:
frame['Ohio']

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

A MultiIndex can be created by itself and then reused; the columns in the preceding
DataFrame with level names could be created like this:

In [63]:
mindex_for_cols = pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
names=['state', 'color'])

In [64]:
mindex_for_rows = pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
names=['key1', 'key2'])

In [68]:
frame2 = pd.DataFrame(np.arange(12).reshape((4, 3)),index= mindex_for_rows,columns=mindex_for_cols)

In [70]:
frame2

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

### Reordering and Sorting Levels

At times you will need to rearrange the order of the levels on an axis or sort the data
by the values in one specific level. The `swaplevel` takes two level numbers or names
and returns a new object with the levels interchanged (but the data is otherwise
unaltered):

In [71]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [72]:
frame.swaplevel('key1', 'key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

`sort_index`, on the other hand, sorts the data using only the values in a single level.
When swapping levels, it’s not uncommon to also use `sort_index` so that the result is
lexicographically sorted by the indicated level:

In [74]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [76]:
frame.sort_index(level=0,ascending= False)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
b    2        9  10       11
     1        6   7        8
a    2        3   4        5
     1        0   1        2

In [77]:
frame.sort_index(level=1, ascending= False)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
b    2        9  10       11
a    2        3   4        5
b    1        6   7        8
a    1        0   1        2

In [78]:
frame.swaplevel(0, 1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

> Data selection performance is much better on hierarchically
indexed objects if the index is lexicographically sorted starting with
the outermost level—that is, the result of calling
`sort_index(level=0)` or `sort_index()`.

### Summary Statistics by Level

Many descriptive and summary statistics on DataFrame and Series have a level
option in which you can specify the level you want to aggregate by on a particular
axis. Consider the above DataFrame; we can aggregate by level on either the rows or
columns like so:

In [79]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [80]:
frame.sum(level='key2')

TypeError: sum() got an unexpected keyword argument 'level'

This is failing because in newer version as of the time of this notebook, this option is deprecated. 

##### ✅ Why you are getting this error

You are seeing:

```
TypeError: sum() got an unexpected keyword argument 'level'
```

because **in modern pandas versions (`>= 1.3` and fully removed later)** the `level=` argument in reduction functions like:

```
frame.sum(level='key2')
```

has been **deprecated and removed.**

Earlier (very old pandas — like in the book/course you are studying), this worked because pandas allowed:

```
DataFrame.sum(level=...)
```

to perform aggregation across a **MultiIndex level.**

That API is now replaced with **`groupby(level=...)`.**

---

##### ✅ Correct Modern Pandas Way (Very Important)

You must now write:

```
frame2.groupby(level='key2').sum()
```

This is the **exact modern equivalent** of:

```
frame2.sum(level='key2')
```

---

##### ✅ Let’s Understand What is Happening Conceptually

Your dataframe:

* Rows → MultiIndex (`key1`, `key2`)
* Columns → MultiIndex (`state`, `color`)

When you do:

```
groupby(level='key2')
```

You are saying:

👉 **“Group rows that have the same `key2` value and then sum them.”**

So rows:

```
(a,1)
(b,1)
```

will be grouped together

and rows:

```
(a,2)
(b,2)
```

will be grouped together

Then sum is computed column-wise.

---

##### ✅ Full Working Example

```
frame2.groupby(level='key2').sum()
```

Result structure:

* Index → only `key2`
* Columns → unchanged MultiIndex columns

---

##### ✅ If You Wanted Column Level Aggregation

Very powerful point (senior-level understanding 🙂)

If you wanted to aggregate **across column MultiIndex level**, you would do:

```
frame2.groupby(level='state', axis=1).sum()
```

So now pandas groups **columns instead of rows.**

---

##### ✅ Why Pandas Removed `level=` from `.sum()`

Architectural reason:

* Old API → hidden grouping behaviour inside reduction
* New API → **explicit grouping step**
* Cleaner mental model
* Consistent with:

```
groupby → aggregate
```

This aligns pandas more with SQL / distributed systems / dataframe engines.

---

##### ⭐ Senior Insight (Very Useful for You in Data Science)

Whenever you see:

```
something.sum(level=...)
```

Immediately translate in your head to:

```
something.groupby(level=...).sum()
```

This will save you **huge debugging time** when reading older books / notebooks / courses.

---


In [81]:
frame2.groupby(level='key2').sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

Under the hood, this utilizes pandas’s groupby machinery, which will be discussed in
more detail later in the book.

### Indexing with a DataFrame’s columns

It’s not unusual to want to use one or more columns from a DataFrame as the row
index; alternatively, you may wish to move the row index into the DataFrame’s columns.
Here’s an example DataFrame:

In [84]:
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two',
                            'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})

In [85]:
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame’s `set_index` function will create a new DataFrame using one or more of
its columns as the index:

In [86]:
frame2 = frame.set_index(['c', 'd'])

In [87]:
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

By default the columns are removed from the DataFrame, though you can leave them
in:

In [88]:
frame.set_index(['c', 'd'], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

`reset_index`, on the other hand, does the opposite of `set_index`; the hierarchical
index levels are moved into the columns:



In [97]:
print("TEST PUBLISH 5")

TEST PUBLISH 5
